In [2]:
import pandas as pd

In [3]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

Saving zepto_v2.csv to zepto_v2.csv


In [4]:
df = pd.read_csv("zepto_v2.csv")

In [5]:
df.head()

,Category,name,mrp,discountPercent,availableQuantity,discountedSellingPrice,weightInGms,outOfStock,quantity
0,Fruits & Vegetables,Onion,2500,16,3,2100,1000,False,1
1,Fruits & Vegetables,Tomato Hybrid,4200,16,3,3500,1000,False,1
2,Fruits & Vegetables,Tender Coconut,5100,15,3,4300,58,False,1
3,Fruits & Vegetables,Coriander Leaves,2000,15,3,1700,100,False,100
4,Fruits & Vegetables,Ladies Finger,1400,14,3,1200,250,False,250


In [6]:
import sqlite3
import pandas as pd


conn = sqlite3.connect("zepto.db")


df.to_sql("zepto", conn, index=False, if_exists="replace")


def sql_query(query):
    return pd.read_sql_query(query, conn)

**DATA EXPLORATION**

In [7]:
sql_query("""
SELECT COUNT(*) FROM zepto
""")

,COUNT(*)
0,3732


In [8]:
sql_query("""
SELECT * FROM zepto
LIMIT 10
""")

,Category,name,mrp,discountPercent,availableQuantity,discountedSellingPrice,weightInGms,outOfStock,quantity
0,Fruits & Vegetables,Onion,2500,16,3,2100,1000,0,1
1,Fruits & Vegetables,Tomato Hybrid,4200,16,3,3500,1000,0,1
2,Fruits & Vegetables,Tender Coconut,5100,15,3,4300,58,0,1
3,Fruits & Vegetables,Coriander Leaves,2000,15,3,1700,100,0,100
4,Fruits & Vegetables,Ladies Finger,1400,14,3,1200,250,0,250
5,Fruits & Vegetables,Potato,3500,17,3,2900,1000,0,1
6,Fruits & Vegetables,Lemon,7500,16,3,6300,200,0,200
7,Fruits & Vegetables,Watermelon,5800,15,3,4900,58,0,1
8,Fruits & Vegetables,Capsicum Green,2300,17,3,1900,250,0,250
9,Fruits & Vegetables,Chilli Green,1900,15,3,1600,100,0,100


In [9]:
sql_query("SELECT * FROM zepto").isnull().sum()

,0
Category,0
name,0
mrp,0
discountPercent,0
availableQuantity,0
discountedSellingPrice,0
weightInGms,0
outOfStock,0
quantity,0


In [10]:
#distinct product categories
sql_query("""
SELECT DISTINCT Category
FROM zepto
ORDER BY Category ASC;
""")

,Category
0,Beverages
1,Biscuits
2,Chocolates & Candies
3,Cooking Essentials
4,"Dairy, Bread & Batter"
5,Fruits & Vegetables
6,Health & Hygiene
7,Home & Cleaning
8,Ice Cream & Desserts
9,"Meats, Fish & Eggs"


In [11]:
#products that are in stock
sql_query("""
SELECT Category, name
FROM zepto
WHERE outOfStock = FALSE;
""")

,Category,name
0,Fruits & Vegetables,Onion
1,Fruits & Vegetables,Tomato Hybrid
2,Fruits & Vegetables,Tender Coconut
3,Fruits & Vegetables,Coriander Leaves
4,Fruits & Vegetables,Ladies Finger
...,...,...
3274,Health & Hygiene,Organic India Brahmi Bottle
3275,Health & Hygiene,Horlicks Protein Plus Chocolate
3276,Health & Hygiene,Dabur Nature Care Isabgol - Double Action
3277,Health & Hygiene,Kapiva Wild Tulsi Giloy Juice |


In [12]:
#products that are out of stock
sql_query("""
SELECT Category, name
FROM zepto
WHERE outOfStock = TRUE;
""")

,Category,name
0,Fruits & Vegetables,Carrot
1,Fruits & Vegetables,Beetroot
2,Fruits & Vegetables,Moong Sprouts
3,Fruits & Vegetables,Safal Frozen Mixed Vegetables
4,Fruits & Vegetables,Safal Frozen Corn
...,...,...
448,Health & Hygiene,Stayfree Secure Dry Cover Extra Large Sanitary...
449,Health & Hygiene,Dabur Honitus Herbal Cough Remedy Ayurvedic Syrup
450,Health & Hygiene,Whisper Bindazzz Night Sanitary Pads XL Plus
451,Health & Hygiene,Fine Life Cotton Balls


In [13]:
#product names present multiple times
sql_query("""
SELECT name, COUNT(*) AS count
FROM zepto
GROUP BY name
HAVING COUNT(*) > 1;
""")

,name,count
0,"""Maggi Magic Cubes, Vegetarian Masala (Free 2 ...",2
1,"""Maggi Magic Cubes, Vegetarian Masala""",2
2,24 Mantra Bura Sugar (Demerara),2
3,24 Mantra Groundnut Oil (Cold Pressed) Pet Bottle,2
4,24 Mantra Mustard Oil (Cold Pressed),2
...,...,...
1209,iD Curd Pouch,2
1210,iD Idli & Dosa Batter,7
1211,iD Malabar Parota,3
1212,iD Natural Paneer,2


 **DATA CLEANING**

In [14]:
sql_query("""
SELECT name, mrp / 100.0 AS mrp_rupees
FROM zepto;
""")

,name,mrp_rupees
0,Onion,25.0
1,Tomato Hybrid,42.0
2,Tender Coconut,51.0
3,Coriander Leaves,20.0
4,Ladies Finger,14.0
...,...,...
3727,Stayfree Secure Dry Cover Extra Large Sanitary...,42.0
3728,Dabur Honitus Herbal Cough Remedy Ayurvedic Syrup,105.0
3729,Whisper Bindazzz Night Sanitary Pads XL Plus,185.0
3730,Fine Life Cotton Balls,60.0


In [15]:
conn.execute("""
UPDATE zepto
SET mrp = mrp / 100.0;
""")

conn.commit()

In [16]:
conn.execute("""
UPDATE zepto
SET discountedSellingPrice = discountedSellingPrice / 100.0;
""")

conn.commit()

**BUSINESS INSIGHT QUERIES**

In [17]:
#Q1 Find the top 10 best value products based on the discount percentage.
sql_query("""
SELECT name, discountPercent
FROM zepto
ORDER BY discountPercent DESC
LIMIT 10;
""")

,name,discountPercent
0,Dukes Waffy Chocolate Wafers,51
1,Dukes Waffy Orange Wafers,51
2,Dukes Waffy Strawberry Wafers,51
3,Chef's Basket Durum Wheat Elbow Pasta,50
4,Chef's Basket Durum Wheat Fusilli Pasta,50
5,Ceres Foods Fish Mustard Instant Liquid Masala,50
6,Ceres Foods Laal Maas Instant Liquid Masala,50
7,Ceres Foods Nalli Nihari Instant Liquid Masala,50
8,Chef's Basket Durum Wheat Penne Pasta,50
9,Chef's Basket Durum Wheat Elbow Pasta,50


In [18]:
#Q2 What are the products having high mrp but are out of stock
sql_query("""
SELECT name, mrp
FROM zepto
WHERE outOfStock = TRUE
ORDER BY mrp DESC;
""")



,name,mrp
0,Patanjali Cow's Ghee,565
1,Patanjali Cow's Ghee,565
2,"MamyPoko Pants Standard Diapers, Extra Large (...",399
3,"MamyPoko Pants Standard Diapers, Extra Large (...",399
4,Aashirvaad Atta With Mutigrains,315
...,...,...
448,Britannia 50-50 Maska Chaska Biscuit,10
449,Sunfeast Mom's Magic Cashew & Almond Biscuit,10
450,Britannia Roll Yo! Strawberry Swiss Roll,10
451,Britannia Roll Yo! Choco Swiss Roll (With Egg),10


In [19]:
#Q3 Calculate the estimated revenue for each category
sql_query("""
SELECT
    Category,
    SUM(discountedSellingPrice * quantity) AS estimated_revenue
FROM zepto
GROUP BY Category
ORDER BY estimated_revenue DESC;
""")

,Category,estimated_revenue
0,Packaged Food,17349917.0
1,Ice Cream & Desserts,17349917.0
2,Chocolates & Candies,17349917.0
3,Munchies,13616226.0
4,Cooking Essentials,13616226.0
5,Personal Care,12614825.0
6,Paan Corner,12614825.0
7,"Meats, Fish & Eggs",3693976.0
8,Home & Cleaning,3678343.0
9,"Dairy, Bread & Batter",3330483.0


In [20]:
#Q4 Find all the products where mrp is greater than Rs. 500 and discount is less than 10%
sql_query("""
SELECT name, mrp, discountPercent
FROM zepto
WHERE mrp > 500 AND discountPercent < 10;
""")

,name,mrp,discountPercent
0,Fortune Sunlite Refined Sunflower (Jar),925,0
1,Amul Pure Ghee (Pouch),505,1
2,Amul Ghee (Tin),520,2
3,Fortune Soyabean Oil,1005,0
4,Dhara Kachi Ghani Mustard Oil Jar,1250,8
...,...,...,...
77,Surf Excel Matic Front Load Detergent Powder,530,4
78,Surf Excel Matic Powder Front Load,810,7
79,Lizol Double Concentrate Disinfectant Floor Cl...,650,8
80,Surf Excel Matic Top Load,720,9


In [21]:
#Q5 Identify the top 5 categories offering the highest average discount percentage
sql_query("""
SELECT
    Category,
    AVG(discountPercent) AS average_discount
FROM zepto
GROUP BY Category
ORDER BY average_discount DESC
LIMIT 5;
""")

,Category,average_discount
0,Fruits & Vegetables,15.462366
1,"Meats, Fish & Eggs",11.031746
2,Packaged Food,8.324742
3,Ice Cream & Desserts,8.324742
4,Chocolates & Candies,8.324742


In [22]:
#Q6 Find the price per gram for products above 100g and sort by best value.
sql_query("""
SELECT
    name,
    weightInGms,
    discountedSellingPrice,
    discountedSellingPrice / weightInGms AS price_per_gram
FROM zepto
WHERE weightInGms > 100
ORDER BY price_per_gram ASC;
""")


,name,weightInGms,discountedSellingPrice,price_per_gram
0,Onion,1000,21.0,0.0
1,Tomato Hybrid,1000,35.0,0.0
2,Ladies Finger,250,12.0,0.0
3,Potato,1000,29.0,0.0
4,Lemon,200,63.0,0.0
...,...,...,...,...
2655,"L'Oreal Paris Casting Creme Gloss Hair Color, ...",160,550.0,3.0
2656,"L'Oreal Paris Excellence Creme Hair Color, 4 N...",172,620.0,3.0
2657,"L'Oreal Paris Excellence Creme Hair Color, 3.1...",159,550.0,3.0
2658,Bombay Shaving Company Sensitive Cartridges,116,375.0,3.0


In [23]:
#Q7 Group the products into categories like low, medium and bulk.
sql_query("""
SELECT
    name,
    weightInGms,
    CASE
        WHEN weightInGms < 100 THEN 'Low'
        WHEN weightInGms <= 500 THEN 'Medium'
        ELSE 'Bulk'
    END AS weight_category
FROM zepto;
""")

,name,weightInGms,weight_category
0,Onion,1000,Bulk
1,Tomato Hybrid,1000,Bulk
2,Tender Coconut,58,Low
3,Coriander Leaves,100,Medium
4,Ladies Finger,250,Medium
...,...,...,...
3727,Stayfree Secure Dry Cover Extra Large Sanitary...,406,Medium
3728,Dabur Honitus Herbal Cough Remedy Ayurvedic Syrup,100,Medium
3729,Whisper Bindazzz Night Sanitary Pads XL Plus,870,Bulk
3730,Fine Life Cotton Balls,50,Low


In [24]:
#Q8 What is the total inventory weight per category?
sql_query("""
SELECT
    Category,
    SUM(weightInGms * quantity) AS total_inventory_weight_gms
FROM zepto
GROUP BY Category
ORDER BY total_inventory_weight_gms DESC;
""")

,Category,total_inventory_weight_gms
0,Munchies,43869767
1,Cooking Essentials,43869767
2,Packaged Food,42891488
3,Ice Cream & Desserts,42891488
4,Chocolates & Candies,42891488
5,Personal Care,21638043
6,Paan Corner,21638043
7,Home & Cleaning,14784782
8,"Dairy, Bread & Batter",9247315
9,Beverages,9247315
